# 📊 Pipeline de Jupyter Notebooks — EDA completo de Olist

**Proyecto Integrador · Tecnicatura en Ciencia de Datos e Inteligencia Artificial**

Este notebook es la **puerta de entrada** del pipeline: un conjunto de **11 cuadernos (00 → 10)** que desarma el script `eda_completo.py` en pasos ordenados y autoexplicados.

**Qué contiene el pipeline**

1. La **unificación** de las 9 tablas crudas de Kaggle (Olist, marketplace brasileño de e-commerce) en una única tabla a nivel de *ítem de pedido*.
2. La **limpieza** del dataset: duplicados, fechas, nulos y coordenadas fuera de rango.
3. La **ingeniería de variables** (features): 9 variables nuevas — tiempos de entrega, envío demorado, distancia cliente-vendedor, volumen, etc.
4. El **análisis exploratorio completo (EDA)**: estadísticas descriptivas, gráficos, análisis de logística/precio/vendedores, mapas, NLP sobre las reseñas en portugués y contraste formal de hipótesis.

**Criterio de explicación** (el mismo que usa el script original): cada celda de código va precedida por un markdown con **Qué hacemos** y **Para qué**, y seguida — cuando corresponde — del **Insight** que deja ese resultado.

## Convenciones del pipeline

- **Cada notebook es autocontenido**: arranca con una *celda estándar* de configuración (imports, rutas, paleta de Olist y, cuando grafica, el estilo de los gráficos). Esa celda se explica celda por celda **una sola vez**, en `01_configuracion_inicial.ipynb`; los demás notebooks la repiten (en versión resumida) para poder correrse por separado.
- **Se ejecutan en orden** (00 → 10): cada notebook consume el archivo que genera el anterior.
- **Artefactos** (archivos intermedios) que encadenan el pipeline:

  | Artefacto | Lo genera | Lo consumen |
  |---|---|---|
  | `data/olist_dataset_unificado.csv` | notebook 02 | notebook 03 |
  | `data/pipeline/01_df_limpio.csv` | notebook 03 | notebook 04 |
  | `data/pipeline/02_df_features.csv` | notebook 04 | notebooks 05 → 10 |

- **Figuras**: cada notebook guarda sus gráficos en `figuras_eda/` con un prefijo propio (`nb06_01.png`, `nb07_02.png`, …), de modo que **no se pisan** con los `fig_XX.png` que genera el script original.
- **Rutas**: el código detecta la raíz del proyecto aunque Jupyter haya arrancado desde la carpeta `notebooks/`, así que da igual desde dónde abras el cuaderno.

## Mapa del pipeline

| # | Notebook | Paso | Sección del script | Entra | Sale |
|---|---|---|---|---|---|
| 00 | `00_presentacion_del_pipeline` | Presentación + verificación del entorno | — | — | diagnóstico |
| 01 | `01_configuracion_inicial` | Configuración compartida (imports, rutas, paleta, estilo) | 0 | — | constantes usadas por todos |
| 02 | `02_unificacion_de_las_9_tablas` | Integración de las 9 tablas relacionadas | 1 | `data/raw/*.csv` | `data/olist_dataset_unificado.csv` |
| 03 | `03_carga_y_limpieza` | Carga + limpieza | 2–3 | unificado | `data/pipeline/01_df_limpio.csv` |
| 04 | `04_feature_engineering` | 9 variables nuevas | 4 | `01_df_limpio.csv` | `data/pipeline/02_df_features.csv` |
| 05 | `05_estadisticas_descriptivas` | Estadísticos y conteos con % | 5 | `02_df_features.csv` | tablas impresas |
| 06 | `06_visualizaciones_exploratorias` | 8 gráficos exploratorios | 6 | `02_df_features.csv` | `figuras_eda/nb06_*.png` |
| 07 | `07_analisis_exhaustivo_logistica` | Logística, precio y vendedores | 7 | `02_df_features.csv` | `figuras_eda/nb07_*.png` |
| 08 | `08_mapa_geografico` | Mapas (Plotly + Folium) y resumen por estado | 8 | `02_df_features.csv` | `figuras_eda/nb08_*.png` |
| 09 | `09_nlp_sobre_las_resenas` | NLP sobre reseñas en portugués | 9 | `02_df_features.csv` | `figuras_eda/nb09_*.png` |
| 10 | `10_contraste_de_hipotesis` | Tests estadísticos + conclusiones | 10 | `02_df_features.csv` | veredictos de hipótesis |

## Cómo ejecutarlo

1. Instalá las dependencias (una sola vez):

   ```bash
   pip install -r requirements.txt
   pip install jupyterlab        # o: pip install notebook
   ```

2. Desde la **raíz del proyecto**, abrí Jupyter:

   ```bash
   jupyter lab
   ```

3. Entrá a la carpeta `notebooks/` y ejecutá los cuadernos **en orden** (`Run All`, de arriba hacia abajo). No hace falta reiniciar el kernel entre notebook y notebook: cada uno arranca de cero con su propia celda de configuración.

4. Para rehacer el pipeline desde cero: borrá (o renombrá) los archivos de `data/pipeline/` y `data/olist_dataset_unificado.csv` y volvé a ejecutar los notebooks 02 → 10.

> **Sobre los datos:** los 9 CSV originales ya están en `data/raw/` (bajados una sola vez de [Kaggle](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce), sin token de API), así que el pipeline corre 100% offline una vez instaladas las librerías.

## Verificación del entorno

**Qué hacemos:** comprobamos que las 9 tablas crudas estén en su lugar, qué artefactos del pipeline existen ya (si ejecutaste algún notebook antes) y qué librerías hay instaladas.

**Para qué:** para detectar en 5 segundos cualquier problema (falta un CSV, falta una librería) **antes** de empezar a ejecutar los pasos pesados del pipeline.

In [1]:
from pathlib import Path

BASE = Path.cwd()
if not (BASE / "data").exists() and (BASE.parent / "data").exists():
    BASE = BASE.parent

RAW_DIR = BASE / "data" / "raw"
PIPELINE_DIR = BASE / "data" / "pipeline"
CSV_UNIFICADO = BASE / "data" / "olist_dataset_unificado.csv"

TABLAS = [
    "olist_orders_dataset.csv",
    "olist_order_items_dataset.csv",
    "olist_order_payments_dataset.csv",
    "olist_order_reviews_dataset.csv",
    "olist_customers_dataset.csv",
    "olist_products_dataset.csv",
    "olist_sellers_dataset.csv",
    "olist_geolocation_dataset.csv",
    "product_category_name_translation.csv",
]

print("Raíz del proyecto:", BASE)
print()
print("1) Tablas crudas (data/raw):")
for t in TABLAS:
    ruta = RAW_DIR / t
    if ruta.exists():
        print(f"   [OK]    {t:42s} {ruta.stat().st_size / 1e6:6.1f} MB")
    else:
        print(f"   [FALTA] {t}")

print()
print("2) Artefactos del pipeline:")
for ruta, origen in [
    (CSV_UNIFICADO, "notebook 02 - unificación"),
    (PIPELINE_DIR / "01_df_limpio.csv", "notebook 03 - limpieza"),
    (PIPELINE_DIR / "02_df_features.csv", "notebook 04 - features"),
]:
    estado = "OK" if ruta.exists() else "falta"
    print(f"   [{estado:4s}] {ruta.name:30s} <- {origen}")

print()
print("3) Librerías del requirements.txt:")
for mod in ["pandas", "numpy", "scipy", "matplotlib", "seaborn",
            "plotly", "folium", "wordcloud", "nltk"]:
    try:
        m = __import__(mod)
        print(f"   [OK]    {mod:11s} {getattr(m, '__version__', '?')}")
    except ImportError:
        print(f"   [FALTA] {mod:11s} -> pip install {mod}")

Raíz del proyecto: d:\Ciencia de Datos\Proyecto Integrador\trabajo_ integrador_versionNati\trabajo_ integrador_versionNati

1) Tablas crudas (data/raw):
   [OK]    olist_orders_dataset.csv                     17.7 MB
   [OK]    olist_order_items_dataset.csv                15.4 MB
   [OK]    olist_order_payments_dataset.csv              5.8 MB
   [OK]    olist_order_reviews_dataset.csv              14.5 MB
   [OK]    olist_customers_dataset.csv                   9.0 MB
   [OK]    olist_products_dataset.csv                    2.4 MB
   [OK]    olist_sellers_dataset.csv                     0.2 MB
   [OK]    olist_geolocation_dataset.csv                61.3 MB
   [OK]    product_category_name_translation.csv         0.0 MB

2) Artefactos del pipeline:
   [OK  ] olist_dataset_unificado.csv    <- notebook 02 - unificación
   [OK  ] 01_df_limpio.csv               <- notebook 03 - limpieza
   [OK  ] 02_df_features.csv             <- notebook 04 - features

3) Librerías del requirements.txt:
  

### Cómo leer esta salida

- **1) Tablas crudas:** las 9 deben figurar en `[OK]`. Si falta alguna, volvé a descargar el dataset de Kaggle (sección *Si el dataset se actualiza* del `README.md`) y reemplazá los CSV de `data/raw/`.
- **2) Artefactos:** en una primera corrida los tres aparecerán como `falta` — se generan solos al ejecutar los notebooks 02, 03 y 04. Si ya aparece alguno en `OK`, ese paso ya se ejecutó.
- **3) Librerías:** todo en `[OK]` → listo para continuar. Si alguna figura en `[FALTA]`, instalá el `requirements.txt` completo (`pip install -r requirements.txt`).

**Siguiente paso:** abrí `01_configuracion_inicial.ipynb` y ejecutalo.